# Alpha191 因子研究 · 本地化适配版

本 Notebook 演示如何使用适配后的 Alpha191 因子库在本地 A 股数据上进行因子计算和回测分析。

## 工作流
1. 使用 `my_utils.fun.read_day_data()` 加载本地日线数据
2. 转换为宽表格式 → 计算 191 个价量因子
3. 使用 `因子回测/alpha.py` 的 `analyze_factor()` 进行 IC 分析和分组收益回测

**核心原则**: 尽可能复用本地已有接口，不重复造轮子。

In [ ]:
# 1. 导入
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# 添加项目根目录
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 导入本包
from 因子回测.alpha_191 import Alpha191Calculator, load_factor_data
from 因子回测.alpha_191.alpha_formulas import Alpha191Formulas

print('✅ 导入成功')

In [ ]:
# 2. 数据加载
# 方式一：使用计算器（推荐）
calc = Alpha191Calculator()
calc.load_data(
    start_date='2024-01-01',
    end_date='2025-07-24',
    min_records=100,  # 过滤上市不足100天的股票
)

# 方式二：手动加载（更灵活，可指定股票列表）
# from alpha191.adapter import load_factor_data
# stock_list = None  # 全部股票，或指定 ['SHSE.600000', 'SZSE.000001']
# data = load_factor_data('2024-01-01', '2025-07-24', stock_list=stock_list)
# formulas = Alpha191Formulas(data)

print(f'交易日: {len(calc.trading_dates)}')
print(f'股票数: {len(calc.stock_codes)}')

In [ ]:
# 3. 查看数据样例
calc.data['close'].iloc[:5, :5]

In [ ]:
# 4. 计算单个因子
# 最新日横截面值
alpha5 = calc.compute(5)
print('Alpha #5 最新日因子值:')
print(alpha5.sort_values(ascending=False).head(10))
print()
print(f'非空值: {alpha5.notna().sum()}/{len(alpha5)}')

In [ ]:
# 查看全时段因子值
alpha5_df = calc.compute_df(5)
print(f'Alpha #5 全时段: shape={alpha5_df.shape}')
alpha5_df.iloc[:5, :5]

In [ ]:
# 5. 批量计算多个因子
# 计算一批有代表性的因子
target_alphas = [1, 2, 5, 10, 41, 55, 101, 150, 191]
results = calc.compute_all(target_alphas)

# 查看结果
factor_df = pd.DataFrame({name: results[name] for name in results})
print('各因子最新日值（前5只股票）:')
factor_df.head()

In [ ]:
# 6. 宽表因子回测
# 使用本地 因子回测/alpha.py 的 analyze_factor() 进行IC分析+分组收益回测
# 这是极简版、纯宽表向量化计算

result = calc.analyze_factor(
    alpha_num=5,
    return_period=5,    # 5日持仓
    adjust_freq=1,      # 每日调仓
    group_num=5,        # 分5组
)

print(f"""
══════ Alpha #5 因子分析摘要 ══════
IC均值:     {result['ic_stats']['ic_mean']:.4f}
IC_IR:      {result['ic_stats']['ic_ir']:.4f}
IC>0占比:   {result['ic_stats']['ic_pos_ratio']:.2%}
RankIC均值: {result['ic_stats']['rank_ic_mean']:.4f}
RankIC_IR:  {result['ic_stats']['rank_ic_ir']:.4f}
""")

In [ ]:
# 7. 对比多个因子的表现
def compare_alphas(calc, alpha_list, return_period=5):
    """对比多个因子的IC表现"""
    rows = []
    for n in alpha_list:
        try:
            r = calc.analyze_factor(n, return_period=return_period, group_num=5)
            s = r['ic_stats']
            rows.append({
                'Alpha': f'#{n}',
                'IC均值': s['ic_mean'],
                'IC_IR': s['ic_ir'],
                'IC>0占比': s['ic_pos_ratio'],
                'RankIC均值': s['rank_ic_mean'],
                'RankIC_IR': s['rank_ic_ir'],
            })
            print(f'✅ Alpha #{n} 完成')
        except Exception as e:
            print(f'❌ Alpha #{n}: {e}')
    
    return pd.DataFrame(rows).round(4)

# 对比一组因子
alpha_batch = [1, 2, 5, 10, 41, 101]
comparison = compare_alphas(calc, alpha_batch, return_period=5)
comparison

In [ ]:
# 8. 查看未实现的alpha（需要行业数据）
print('需要行业数据才能计算的alpha:')
ni = calc.get_not_implemented()
print(f'共 {len(ni)} 个: {ni}')
print()
print('需要补充的数据: 行业分类 (中信一级/申万一级)')

In [ ]:
# 9. 获取宽表格式 -> 用于 因子回测/alpha.py 的 analyze_ic()
factor_wide = calc.get_factor_wide(5)
print('factor_wide 格式:')
print(factor_wide.head())
print()
print('可直接传给 analyze_ic(factor_data=factor_wide, stock_data=...)')

In [ ]:
# 10. 进阶：自定义分析
# 如果你想手动控制每个步骤（更灵活）
from 因子回测.alpha_191.adapter import LocalDataAdapter
from 因子回测.alpha_191.alpha_formulas import Alpha191Formulas

# 指定股票列表
custom_stocks = [
    'SHSE.600000',  # 浦发银行
    'SHSE.600036',  # 招商银行
    'SHSE.601166',  # 兴业银行
    'SHSE.600030',  # 中信证券
    'SHSE.601318',  # 中国平安
]

adapter = LocalDataAdapter()
custom_data = adapter.load_data(
    start_date='2025-01-01',
    end_date='2025-07-24',
    stock_list=custom_stocks,
    min_records=50,
)

formulas = Alpha191Formulas(custom_data)
for n in [5, 10, 41, 101]:
    val = getattr(formulas, f'alpha_{n:03d}')()
    print(f'Alpha #{n}: {val.to_dict()}')

## 小结

### 已完成的功能
- ✅ 数据适配器（`adapter.py`）：使用 `my_utils.fun.read_day_data()` 读取本地数据
- ✅ Alpha 公式（`alpha_formulas.py`）：191个因子的计算公式（约15个需行业数据）
- ✅ 计算器（`calculator.py`）：统一接口 + 本地回测框架集成
- ✅ IC分析/分组收益回测：使用 `因子回测/alpha.py` 的现有接口

### 待扩展
- 行业数据接入 → 补齐 Alpha #48, 56, 58-59, 63, 67, 69-70, 76, 79-80, 82, 87, 89-91, 93, 97, 100
- 市值数据 → 补齐 Alpha #56
- 大样本多周期因子扫描